In [1]:
import pandas as pd 
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

orders.head()

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


In [2]:
orders.head(10)

orders.isna().sum()

orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [3]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


In [4]:
orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [5]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - 
    orders["order_purchase_timestamp"]
).dt.days

orders[["order_status", "delivery_time_days"]].head(10)


,order_status,delivery_time_days
0,delivered,8.0
1,delivered,13.0
2,delivered,9.0
3,delivered,13.0
4,delivered,2.0
5,delivered,16.0
6,invoiced,NaN
7,delivered,9.0
8,delivered,9.0
9,delivered,18.0


In [6]:
delivered_orders = orders[orders["order_status"] == "delivered"]

In [7]:
delivered_orders["delivery_time_days"].describe()

count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_time_days, dtype: float64

In [8]:
delivered_orders["delivery_time_days"].quantile(0.95)


np.float64(29.0)

In [9]:
analytics_orders = delivered_orders[[
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_time_days"
]]


In [10]:
import os

os.makedirs("../data/analytics", exist_ok=True)


In [11]:
analytics_orders.to_parquet(
    "../data/analytics/orders_delivery_analytics.parquet",
    index=False
)


In [14]:
import pandas as pd

df = pd.read_parquet("../data/analytics/orders_delivery_analytics.parquet")

df.head(10)

df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 5 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  str           
 1   customer_id                    96478 non-null  str           
 2   order_purchase_timestamp       96478 non-null  datetime64[us]
 3   order_delivered_customer_date  96470 non-null  datetime64[us]
 4   delivery_time_days             96470 non-null  float64       
dtypes: datetime64[us](2), float64(1), str(2)
memory usage: 9.6 MB


,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days
count,96478,96470,96470.000000
mean,2018-01-01 23:29:31.939913,2018-01-14 12:41:33.581683,12.093604
min,2016-09-15 12:16:38,2016-10-11 13:46:32,0.000000
25%,2017-09-14 09:00:23.250000,2017-09-25 22:15:09.500000,6.000000
50%,2018-01-20 19:45:45,2018-02-02 19:32:21,10.000000
75%,2018-05-05 18:54:47,2018-05-15 22:54:48.500000,15.000000
max,2018-08-29 15:00:37,2018-10-17 13:22:46,209.000000
std,NaN,NaN,9.551380
